Hardware:
- MSI Katana
- Intel Core i7-10750H
- NVIDIA GeForce RTX 3050 Laptop GPU

Firstly, we import all code dependencies that will be helpful later on the training process

In [1]:
import pandas as pd
import numpy as np
import os

# Plain Text Generation

In this section we generate a plain text containing all Python scripts included in the datalake. This will be useful in the next section, in order to properly train our gpt-2 fine tuned model

In [2]:
def read_file(path):
  try:
      with open(path) as f:
        for line in f.readlines():
          if line[:6] != "<body>":
            return line
  except:
    print(path)
    return ""

  return ""


def read_files(dir_path):
  path_list = os.listdir(dir_path)
  content = ""

  for path in path_list:
    content += read_file(dir_path + "/" + path)

  return content

In [3]:
datalake_path = "datalake"

In [16]:
text_data = read_files(datalake_path)

datalake/https___github.com_Significant-Gravitas_AutoGPT_blob_master_autogpt_agbenchmark_config_analyze_reports.py
datalake/https___github.com_Significant-Gravitas_AutoGPT_blob_master_autogpt_autogpt_app_telemetry.py
datalake/https___github.com_Significant-Gravitas_AutoGPT_blob_master_autogpt_scripts_git_log_to_release_notes.py
datalake/https___github.com_Significant-Gravitas_AutoGPT_blob_master_benchmark_reports_format.py
datalake/https___github.com_Significant-Gravitas_AutoGPT_blob_master_cli.py
datalake/https___github.com_Significant-Gravitas_AutoGPT_blob_master_forge_forge_utils_test_url_validator.py
datalake/https___github.com_TheAlgorithms_Python_blob_master_dynamic_programming_combination_sum_iv.py
datalake/https___github.com_TheAlgorithms_Python_blob_master_machine_learning_logistic_regression.py
datalake/https___github.com_TheAlgorithms_Python_blob_master_machine_learning_polynomial_regression.py
datalake/https___github.com_TheAlgorithms_Python_blob_master_physics_casimir_effe

Now we split the plain text into train and test data, and store it on drive

In [19]:
train_text = text_data[:int(0.8*len(text_data))]
test_text = text_data[int(0.8*len(text_data)):]

In [20]:
path = ""

with open(path + "train_text.txt", "w") as f:
  f.write(train_text)

with open(path + "test_text.txt", "w") as f:
  f.write(test_text)

# Retraining GPT-2 (Fine tuning)

In this section, we will fine tune GPT-2 using python scripts, all of them obtained via **GitHub Scrapper for RePylot**

In [2]:
from transformers import TextDataset, DataCollatorForLanguageModeling
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import Trainer, TrainingArguments

import torch

In [3]:
def load_dataset(file_path, tokenizer, block_size = 128):
    dataset = TextDataset(
        tokenizer = tokenizer,
        file_path = file_path,
        block_size = block_size,
    )
    return dataset


def load_data_collator(tokenizer, mlm = False):
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=mlm,
    )
    return data_collator

In [4]:
def train(train_file_path,model_name,
          output_dir,
          overwrite_output_dir,
          per_device_train_batch_size,
          num_train_epochs,
          save_steps):
  tokenizer = GPT2Tokenizer.from_pretrained(model_name)
  tokenizer.save_pretrained(output_dir)

  train_dataset = load_dataset(train_file_path, tokenizer)
  data_collator = load_data_collator(tokenizer)

  model = GPT2LMHeadModel.from_pretrained(model_name)
  
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)
  model.save_pretrained(output_dir)

  training_args = TrainingArguments(
          output_dir=output_dir,
          overwrite_output_dir=overwrite_output_dir,
          per_device_train_batch_size=per_device_train_batch_size,
          num_train_epochs=num_train_epochs,
          save_steps=save_steps,
      )
  
  print(model.device)
  trainer = Trainer(
          model=model,
          args=training_args,
          data_collator=data_collator,
          train_dataset=train_dataset,
  )

  trainer.train()
  trainer.save_model()

In [5]:
train_file_path = "train_text.txt"
model_name = 'gpt2'

output_dir = 'models/custom_gpt2'
overwrite_output_dir = True
per_device_train_batch_size = 8
num_train_epochs = 5
save_steps = 0

In [6]:
train(
    train_file_path=train_file_path,
    model_name=model_name,
    output_dir=output_dir,
    overwrite_output_dir=overwrite_output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    num_train_epochs=num_train_epochs,
    save_steps=save_steps
)

C:\Users\carde\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
C:\Users\carde\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\transformers\data\datasets\language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


cuda:0


 19%|█▉        | 500/2595 [12:59<59:35,  1.71s/it]  

{'loss': 2.4484, 'grad_norm': 4.351332664489746, 'learning_rate': 4.036608863198459e-05, 'epoch': 0.96}


 39%|███▊      | 1000/2595 [28:28<44:36,  1.68s/it] 

...

{'loss': 1.8208, 'grad_norm': 4.037161350250244, 'learning_rate': 1.1464354527938344e-05, 'epoch': 3.85}


 96%|█████████▋| 2500/2595 [1:11:35<02:36,  1.64s/it]

{'loss': 1.7593, 'grad_norm': 3.5862619876861572, 'learning_rate': 1.8304431599229288e-06, 'epoch': 4.82}


100%|██████████| 2595/2595 [1:14:34<00:00,  1.72s/it]


{'train_runtime': 4474.9277, 'train_samples_per_second': 4.637, 'train_steps_per_second': 0.58, 'train_loss': 1.999113714258565, 'epoch': 5.0}


# Model Evaluation

We can now proceed testing the model we have just trained

In [22]:
def load_model(model_path):
    model = GPT2LMHeadModel.from_pretrained(model_path)
    return model


def load_tokenizer(tokenizer_path):
    tokenizer = GPT2Tokenizer.from_pretrained(tokenizer_path)
    return tokenizer


def generate_text(model_path, sequence, extra_length):
    model = load_model(model_path).to(torch.device("cuda"))
    tokenizer = load_tokenizer(model_path)

    ids = tokenizer.encode(f'{sequence}', return_tensors='pt').to(torch.device("cuda"))
    final_outputs = model.generate(
        ids,
        do_sample=True,
        max_length=len(ids) + extra_length,
        pad_token_id=model.config.eos_token_id,
        top_k=50,
        top_p=0.95,
    )

    print(tokenizer.decode(final_outputs[0], skip_special_tokens=True))

Feel free to modify `sequence` variable in order to test the model yourself

In [25]:
sequence = "while (t"
generate_text(output_dir, sequence, extra_length=20)

while (t.key == key_count or t.key == key_count &lt;=


If we now compare the obtained result with the original GPT2 model output, we can appreciate how a better response is achieved by our fine tuned transformer. Moreover, the code generated by RePylot is indeed a Python code. Meanwhile, original GPT2 generated code in other language, which possibly doesn't even exist

In [18]:
from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2', device=torch.device("cuda"))

C:\Users\carde\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [19]:
generator(sequence, max_length=30, num_return_sequences=1)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'while (t == null || t.type!= string? "") { return t; } return null; }\n\nWhat if all this code'}]

Other examples are the following

In [36]:
sequence = "for i"

print("RePylot generation:")
generate_text(output_dir, sequence, extra_length=20)

print("\nGPT-2 generation:")
generator(sequence, max_length=30, num_return_sequences=1)

RePylot generation:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


for i in range(len(data)): temp = data[i] * len(data)

GPT-2 generation:


[{'generated_text': 'for i and m are not equals;\n\nto a lesser extent, the power of all its power may be reduced by its inferior powers. But'}]

In [37]:
sequence = "from matplotlib"

print("RePylot generation:")
generate_text(output_dir, sequence, extra_length=20)

print("\nGPT-2 generation:")
generator(sequence, max_length=30, num_return_sequences=1)

RePylot generation:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


from matplotlib.datasets import matplotlib.pyplot as plt from datac

GPT-2 generation:


[{'generated_text': "from matplotlib.python and matplotlib.diamond, and they don't always support the same python_module:\n\n>>> from"}]

Note that these results have been obtained fine tuning GPT-2 in only 5 epochs. Due to the impresive results, we can expect even better results by increasing the number of epochs. Thus, we resume the training process

In [ ]:
def train(train_file_path,model_name,
          output_dir,
          overwrite_output_dir,
          per_device_train_batch_size,
          num_train_epochs,
          save_steps):
  tokenizer = GPT2Tokenizer.from_pretrained(model_name)
  tokenizer.save_pretrained(output_dir)

  train_dataset = load_dataset(train_file_path, tokenizer)
  data_collator = load_data_collator(tokenizer)

  model = GPT2LMHeadModel.from_pretrained(model_name)
  
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)
  model.save_pretrained(output_dir)

  training_args = TrainingArguments(
          output_dir=output_dir,
          overwrite_output_dir=overwrite_output_dir,
          per_device_train_batch_size=per_device_train_batch_size,
          num_train_epochs=num_train_epochs,
          save_steps=save_steps,
      )
  
  print(model.device)
  trainer = Trainer(
          model=model,
          args=training_args,
          data_collator=data_collator,
          train_dataset=train_dataset,
  )

  trainer.train()
  trainer.save_model()

In [38]:
train_file_path = "train_text.txt"
model_name = 'models/custom_gpt2'

output_dir = 'models/custom_gpt2_10'
overwrite_output_dir = False
per_device_train_batch_size = 8
num_train_epochs = 5
save_steps = 0

In [39]:
train(
    train_file_path=train_file_path,
    model_name=model_name,
    output_dir=output_dir,
    overwrite_output_dir=overwrite_output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    num_train_epochs=num_train_epochs,
    save_steps=save_steps
)

C:\Users\carde\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\transformers\data\datasets\language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


cuda:0


  4%|▎         | 94/2595 [04:24<2:02:42,  2.94s/it]

KeyboardInterrupt: 